In [ ]:
#@title Dane studenta
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_tego_notebooka = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_tego_notebooka,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)


# BSM L02 — Podstawowe pojęcia kryptologii, poufność danych, terminologia, proste szyfry

## Co było na poprzednich zajęciach
W L01 pracowaliśmy na triadzie **AIC** i podstawach modelowania zagrożeń.
W praktyce najczęściej analizowaliśmy przepływy, ryzyka i incydenty.

## Co robimy teraz (L02)
Skupiamy się głównie na **C (Confidentiality)**:
- słownik kryptologii,
- model szyfrowania (wiadomość, klucz, szyfrogram, przeciwnik),
- proste szyfry klasyczne,
- podstawy kryptoanalizy.

## Zasady pracy
1. Nie zmieniaj danych wejściowych podanych przy zadaniu.
2. `final_answer` ma być stringiem i mieć dokładnie wymagany format.
3. Jeśli zadanie mówi „piszesz samodzielnie”, implementujesz funkcję od zera.
4. Jeśli zadanie mówi „użyj funkcji z poprzedniego zadania”, korzystasz z wcześniej napisanej implementacji.


In [ ]:
import json
import hashlib

def canonical_json(obj):
    return json.dumps(obj, sort_keys=True, ensure_ascii=False)

def sha256_text(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

answers = {}

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    wyslij_odpowiedz(task_id, final_answer)


# B01 — Terminologia: elementy modelu szyfrowania

## Co implementujesz samodzielnie
```python
def klasyfikuj_pojecia(items: list[str]) -> str
```

## Jak napisać
1. Dla każdego hasła przypisz jedną etykietę: `ALFABET`, `WIADOMOSC`, `KLUCZ`, `SZYFROGRAM`, `ROLA`.
2. Zwróć wynik w formacie `H1:ETYKIETA,H2:ETYKIETA,...`.

## Co oddajesz
```python
items_B01 = ["A-Z", "tekst jawny", "k=7", "QEB NRFZH", "nadawca", "przeciwnik"]
final_answer = klasyfikuj_pojecia(items_B01)
```


### Wyjaśnienie (teacher)
Rozwiązanie referencyjne jest poniżej. Zachowaj kolejność wykonywania komórek i sprawdź zgodność formatu `final_answer`.


In [ ]:
def klasyfikuj_pojecia(items):
    mapping = {
        "A-Z": "ALFABET",
        "tekst jawny": "WIADOMOSC",
        "k=7": "KLUCZ",
        "QEB NRFZH": "SZYFROGRAM",
        "nadawca": "ROLA",
        "przeciwnik": "ROLA",
    }
    return ",".join(f"{x}:{mapping[x]}" for x in items)

items_B01 = ["A-Z", "tekst jawny", "k=7", "QEB NRFZH", "nadawca", "przeciwnik"]
final_answer = klasyfikuj_pojecia(items_B01)
final_answer


In [ ]:
zapisz_i_wyslij("B01", str(final_answer))


# B02 — Szyfr Cezara: implementacja

## Co implementujesz samodzielnie
```python
def normalize_pl_text(text: str) -> str

def caesar_encrypt(text: str, shift: int, alphabet: str = "ABCDEFGHIJKLMNOPQRSTUVWXYZ") -> str

def caesar_decrypt(text: str, shift: int, alphabet: str = "ABCDEFGHIJKLMNOPQRSTUVWXYZ") -> str
```

## Jak napisać
1. Zamień polskie znaki na ASCII i tekst na wielkie litery.
2. Szyfruj przez przesunięcie modulo długość alfabetu.
3. Deszyfruj przez przeciwne przesunięcie.

## Co oddajesz
```python
plain_B02 = "Poufność danych mobilnych jest kluczowa"
shift_B02 = 9
cipher_B02 = caesar_encrypt(normalize_pl_text(plain_B02), shift_B02)
final_answer = cipher_B02
```


### Wyjaśnienie (teacher)
Rozwiązanie referencyjne jest poniżej. Zachowaj kolejność wykonywania komórek i sprawdź zgodność formatu `final_answer`.


In [ ]:
import unicodedata

ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
_PL_MAP = str.maketrans({
    "Ą":"A","Ć":"C","Ę":"E","Ł":"L","Ń":"N","Ó":"O","Ś":"S","Ż":"Z","Ź":"Z",
    "ą":"A","ć":"C","ę":"E","ł":"L","ń":"N","ó":"O","ś":"S","ż":"Z","ź":"Z"
})

def normalize_pl_text(text):
    text = text.translate(_PL_MAP)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text.upper()

def caesar_encrypt(text, shift, alphabet=ALPHABET):
    idx = {c:i for i,c in enumerate(alphabet)}
    n = len(alphabet)
    out = []
    for ch in text:
        if ch in idx:
            out.append(alphabet[(idx[ch] + shift) % n])
        else:
            out.append(ch)
    return "".join(out)

def caesar_decrypt(text, shift, alphabet=ALPHABET):
    return caesar_encrypt(text, -shift, alphabet)

plain_B02 = "Poufność danych mobilnych jest kluczowa"
shift_B02 = 9
cipher_B02 = caesar_encrypt(normalize_pl_text(plain_B02), shift_B02)
final_answer = cipher_B02
final_answer


In [ ]:
zapisz_i_wyslij("B02", str(final_answer))


# B03 — Brute force Cezara

## Skąd funkcje
`normalize_pl_text`, `caesar_decrypt` bierzesz z **B02**.

## Co implementujesz samodzielnie
```python
def score_tekst_pl(text: str) -> int

def auto_break_caesar(cipher: str) -> tuple[str, int]
```

## Jak napisać
1. Sprawdź wszystkie przesunięcia 0..25.
2. Oceniaj kandydatów przez słowa kluczowe.
3. Zwróć najlepszy `(tekst, shift)`.

## Co oddajesz
```python
cipher_B03 = "IFSJ PQNJSYF RTGNQSDHM IFSDHM"
best_text_B03, best_shift_B03 = auto_break_caesar(cipher_B03)
final_answer = f"{best_shift_B03}|{best_text_B03}"
```


### Wyjaśnienie (teacher)
Rozwiązanie referencyjne jest poniżej. Zachowaj kolejność wykonywania komórek i sprawdź zgodność formatu `final_answer`.


In [ ]:
def score_tekst_pl(text):
    t = normalize_pl_text(text)
    words = ["DANE", "KLIENT", "MOBILNYCH", "POUFNOSC", "SYSTEM"]
    return sum(t.count(w) * (3 + len(w)//3) for w in words) + t.count(" ")

def auto_break_caesar(cipher):
    best = ("", 0, -1)
    for s in range(26):
        cand = caesar_decrypt(cipher, s)
        sc = score_tekst_pl(cand)
        if sc > best[2]:
            best = (cand, s, sc)
    return best[0], best[1]

cipher_B03 = "IFSJ PQNJSYF RTGNQSDHM IFSDHM"
best_text_B03, best_shift_B03 = auto_break_caesar(cipher_B03)
final_answer = f"{best_shift_B03}|{best_text_B03}"
final_answer


In [ ]:
zapisz_i_wyslij("B03", str(final_answer))


# B04 — Częstości liter i indeks koincydencji

## Skąd funkcje
`normalize_pl_text` bierzesz z **B02**.

## Co implementujesz samodzielnie
```python
def letter_freq(text: str, alphabet: str = "ABCDEFGHIJKLMNOPQRSTUVWXYZ") -> dict[str, float]

def index_of_coincidence(text: str, alphabet: str = "ABCDEFGHIJKLMNOPQRSTUVWXYZ") -> float
```

## Co oddajesz
```python
text_B04 = normalize_pl_text("Bezpieczenstwo aplikacji mobilnej wymaga ochrony danych")
freq_B04 = letter_freq(text_B04)
top3_B04 = sorted(freq_B04.items(), key=lambda kv: (-kv[1], kv[0]))[:3]
ioc_B04 = index_of_coincidence(text_B04)
final_answer = f"{top3_B04}|{ioc_B04:.4f}"
```


### Wyjaśnienie (teacher)
Rozwiązanie referencyjne jest poniżej. Zachowaj kolejność wykonywania komórek i sprawdź zgodność formatu `final_answer`.


In [ ]:
from collections import Counter

def letter_freq(text, alphabet=ALPHABET):
    f = [ch for ch in text if ch in alphabet]
    n = len(f)
    cnt = Counter(f)
    if n == 0:
        return {ch: 0.0 for ch in alphabet}
    return {ch: cnt.get(ch, 0) / n for ch in alphabet}

def index_of_coincidence(text, alphabet=ALPHABET):
    f = [ch for ch in text if ch in alphabet]
    n = len(f)
    if n < 2:
        return 0.0
    cnt = Counter(f)
    return sum(v*(v-1) for v in cnt.values()) / (n*(n-1))

text_B04 = normalize_pl_text("Bezpieczenstwo aplikacji mobilnej wymaga ochrony danych")
freq_B04 = letter_freq(text_B04)
top3_B04 = sorted(freq_B04.items(), key=lambda kv: (-kv[1], kv[0]))[:3]
ioc_B04 = index_of_coincidence(text_B04)
final_answer = f"{top3_B04}|{ioc_B04:.4f}"
final_answer


In [ ]:
zapisz_i_wyslij("B04", str(final_answer))


# B05 — Szyfr podstawieniowy

## Skąd funkcje
`normalize_pl_text` bierzesz z **B02**.

## Co implementujesz samodzielnie
```python
def substitution_encrypt(text: str, alphabet: str, key: str) -> str

def substitution_decrypt(cipher: str, alphabet: str, key: str) -> str
```

## Co oddajesz
```python
alphabet_B05 = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
key_B05 = "QWERTYUIOPASDFGHJKLZXCVBNM"
plain_B05 = normalize_pl_text("Model zagrozen aplikacji mobilnej")
cipher_B05 = substitution_encrypt(plain_B05, alphabet_B05, key_B05)
final_answer = cipher_B05
```


### Wyjaśnienie (teacher)
Rozwiązanie referencyjne jest poniżej. Zachowaj kolejność wykonywania komórek i sprawdź zgodność formatu `final_answer`.


In [ ]:
def substitution_encrypt(text, alphabet, key):
    t = normalize_pl_text(text)
    mp = {a:b for a,b in zip(alphabet, key)}
    return "".join(mp.get(ch, ch) for ch in t)

def substitution_decrypt(cipher, alphabet, key):
    inv = {b:a for a,b in zip(alphabet, key)}
    return "".join(inv.get(ch, ch) for ch in normalize_pl_text(cipher))

alphabet_B05 = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
key_B05 = "QWERTYUIOPASDFGHJKLZXCVBNM"
plain_B05 = normalize_pl_text("Model zagrozen aplikacji mobilnej")
cipher_B05 = substitution_encrypt(plain_B05, alphabet_B05, key_B05)
final_answer = cipher_B05
final_answer


In [ ]:
zapisz_i_wyslij("B05", str(final_answer))


# B06 — Szyfr transpozycyjny kolumnowy

## Skąd funkcje
`normalize_pl_text` bierzesz z **B02**.

## Co implementujesz samodzielnie
```python
def columnar_encrypt(text: str, key: str) -> str

def columnar_decrypt(cipher: str, key: str) -> str
```

## Co oddajesz
```python
plain_B06 = normalize_pl_text("Poufne dane aplikacji mobilnej")
key_B06 = "MOBILE"
cipher_B06 = columnar_encrypt(plain_B06, key_B06)
final_answer = cipher_B06
```


### Wyjaśnienie (teacher)
Rozwiązanie referencyjne jest poniżej. Zachowaj kolejność wykonywania komórek i sprawdź zgodność formatu `final_answer`.


In [ ]:
def _column_order(key):
    return sorted(range(len(key)), key=lambda i: (key[i], i))

def columnar_encrypt(text, key):
    t = normalize_pl_text(text)
    k = normalize_pl_text(key)
    cols = len(k)
    rows = (len(t) + cols - 1) // cols
    padded = t + "X" * (rows * cols - len(t))
    grid = [padded[i*cols:(i+1)*cols] for i in range(rows)]
    order = _column_order(k)
    return "".join("".join(grid[r][c] for r in range(rows)) for c in order)

def columnar_decrypt(cipher, key):
    c = normalize_pl_text(cipher)
    k = normalize_pl_text(key)
    cols = len(k)
    rows = len(c) // cols
    order = _column_order(k)
    col_data = {}
    pos = 0
    for col in order:
        col_data[col] = c[pos:pos+rows]
        pos += rows
    out = []
    for r in range(rows):
        for col in range(cols):
            out.append(col_data[col][r])
    return "".join(out)

plain_B06 = normalize_pl_text("Poufne dane aplikacji mobilnej")
key_B06 = "MOBILE"
cipher_B06 = columnar_encrypt(plain_B06, key_B06)
final_answer = cipher_B06
final_answer


In [ ]:
zapisz_i_wyslij("B06", str(final_answer))


# B07 — Koszt ataku siłowego (przestrzeń kluczy)

## Co implementujesz samodzielnie
```python
def raport_bruteforce() -> str
```

## Jak napisać
1. Policz czas ataku dla Cezara, 40-bit, 56-bit i 128-bit.
2. Załóż tempo `2 * 10^9` kluczy/s.
3. Wynik w godzinach, format: `cezar|40b|56b|128b`.

## Co oddajesz
```python
final_answer = raport_bruteforce()
```


### Wyjaśnienie (teacher)
Rozwiązanie referencyjne jest poniżej. Zachowaj kolejność wykonywania komórek i sprawdź zgodność formatu `final_answer`.


In [ ]:
def raport_bruteforce():
    speed = 2 * 10**9
    cases = [("cezar", 26), ("40b", 2**40), ("56b", 2**56), ("128b", 2**128)]

    def fmt(h):
        if h > 1e9:
            import math
            e = int(math.floor(math.log10(h)))
            m = h / (10**e)
            return f"{m:.2f}e{e}h"
        return f"{h:.2f}h"

    out = []
    for name, tries in cases:
        h = tries / speed / 3600
        out.append(f"{name}:{fmt(h)}")
    return "|".join(out)

final_answer = raport_bruteforce()
final_answer


In [ ]:
zapisz_i_wyslij("B07", str(final_answer))


# B08 — One-time pad i błąd ponownego użycia klucza

## Co implementujesz samodzielnie
```python
def xor_bytes(a: bytes, b: bytes) -> bytes

def recover_prefix_from_reused_key(c1: bytes, c2: bytes, known_prefix: bytes) -> bytes
```

## Co oddajesz
```python
m1_B08 = b"MOBILE_PAYLOAD:USER=ALFA;AMOUNT=1200"
m2_B08 = b"MOBILE_PAYLOAD:USER=BETA;AMOUNT=9800"
key_B08 = bytes([57,12,140,125,114,71,52,44,216,16,15,47,111,119,13,101,214,112,229,142,3,81,216,174,142,79,110,172,52,47,194,49,183,176,135,22,235])

c1 = xor_bytes(m1_B08, key_B08)
c2 = xor_bytes(m2_B08, key_B08)
known_prefix = b"MOBILE_PAYLOAD:USER="
recovered = recover_prefix_from_reused_key(c1, c2, known_prefix)
final_answer = recovered.decode("ascii")
```


### Wyjaśnienie (teacher)
Rozwiązanie referencyjne jest poniżej. Zachowaj kolejność wykonywania komórek i sprawdź zgodność formatu `final_answer`.


In [ ]:
def xor_bytes(a, b):
    return bytes(x ^ y for x, y in zip(a, b))

def recover_prefix_from_reused_key(c1, c2, known_prefix):
    m1_xor_m2 = xor_bytes(c1, c2)
    return xor_bytes(m1_xor_m2[:len(known_prefix)], known_prefix)

m1_B08 = b"MOBILE_PAYLOAD:USER=ALFA;AMOUNT=1200"
m2_B08 = b"MOBILE_PAYLOAD:USER=BETA;AMOUNT=9800"
key_B08 = bytes([57,12,140,125,114,71,52,44,216,16,15,47,111,119,13,101,214,112,229,142,3,81,216,174,142,79,110,172,52,47,194,49,183,176,135,22,235])

c1 = xor_bytes(m1_B08, key_B08)
c2 = xor_bytes(m2_B08, key_B08)
known_prefix = b"MOBILE_PAYLOAD:USER="
recovered = recover_prefix_from_reused_key(c1, c2, known_prefix)
final_answer = recovered.decode("ascii")
final_answer


In [ ]:
zapisz_i_wyslij("B08", str(final_answer))


# B09 — Modele ataków: ciphertext-only, known-plaintext, chosen-plaintext

## Co implementujesz samodzielnie
```python
def klasyfikuj_ataki() -> str
```

## Jak napisać
Przypisz właściwy model ataku (`CO`, `KP`, `CP`) do opisów.

## Co oddajesz
```python
# format: 1:CO,2:KP,3:CP,4:KP
final_answer = klasyfikuj_ataki()
```

Opis przypadków:
1. Atakujący ma tylko zbiór szyfrogramów.
2. Atakujący zna pary: tekst jawny i odpowiadający szyfrogram.
3. Atakujący może wybierać wiadomości do zaszyfrowania.
4. Atakujący zna fragmenty standardowych nagłówków i ich szyfrogramy.


### Wyjaśnienie (teacher)
Rozwiązanie referencyjne jest poniżej. Zachowaj kolejność wykonywania komórek i sprawdź zgodność formatu `final_answer`.


In [ ]:
def klasyfikuj_ataki():
    mapping = {1: "CO", 2: "KP", 3: "CP", 4: "KP"}
    return ",".join(f"{k}:{v}" for k, v in mapping.items())

final_answer = klasyfikuj_ataki()
final_answer


In [ ]:
zapisz_i_wyslij("B09", str(final_answer))


# B10 — Mini model poufności dla aplikacji mobilnej

## Co implementujesz samodzielnie
```python
def build_mobile_confidentiality_model() -> str
```

## Jak napisać
Zwróć JSON-string z polami:
- `assets` (min. 4),
- `adversary_capabilities` (min. 4),
- `controls` (min. 6),
- `key_management_notes` (min. 3 punkty),
- `residual_risk`.

## Co oddajesz
```python
final_answer = build_mobile_confidentiality_model()
```


### Wyjaśnienie (teacher)
Rozwiązanie referencyjne jest poniżej. Zachowaj kolejność wykonywania komórek i sprawdź zgodność formatu `final_answer`.


In [ ]:
def build_mobile_confidentiality_model():
    model = {
        "assets": [
            "tokeny sesyjne użytkowników",
            "dane płatności w API",
            "lokalny cache aplikacji",
            "sekrety konfiguracyjne klienta mobilnego"
        ],
        "adversary_capabilities": [
            "podsłuch kanału publicznego",
            "modyfikacja ruchu (MITM)",
            "wstrzykiwanie wiadomości do API",
            "analiza przechwyconych szyfrogramów"
        ],
        "controls": [
            "TLS + pinning certyfikatu",
            "szyfrowanie danych w spoczynku",
            "rotacja kluczy i tokenów",
            "ograniczenie logowania danych wrażliwych",
            "uwierzytelnienie żądań podpisem",
            "monitoring anomalii i alerty"
        ],
        "key_management_notes": [
            "klucze sesyjne krótkotrwałe",
            "bezpieczne przechowywanie kluczy w keystore",
            "zakaz ponownego użycia materiału kluczowego OTP"
        ],
        "residual_risk": "Ryzyko błędów implementacyjnych i wycieku po stronie klienta pozostaje średnie."
    }
    return canonical_json(model)

final_answer = build_mobile_confidentiality_model()
final_answer


In [ ]:
zapisz_i_wyslij("B10", str(final_answer))
